# iSwing+ v9

Single unified model predicting xwOBAcon from swing mechanics, adjustability, and player traits.

**Architecture:** One XGBRegressor. No A/B split.

**Features:**
- Raw mechanics: bat_speed, swing_length
- Speed context: speed_over_expected, speed_vs_location
- Swing adjustability: aa_vs_optimal, aa_adjustment, tilt_for_height, direction_from_optimal, length_for_location
- Effort: effort_level (bat speed vs personal 90th pct)
- Hitter traits: hard_swing_contact, aa_adaptability, dir_adaptability

**Removed:** All pitch-type features, raw attack_angle/tilt/direction, squared_up_rate, location as standalone features.

Run cells 1-8 in order.

## 1 — Install Packages

In [ ]:
!pip install -q pybaseball pandas numpy scikit-learn xgboost joblib streamlit plotly pyngrok
print('All packages installed.')

## 2 — Scrape (skip if CSV exists)

In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from pybaseball import statcast
from datetime import datetime, timedelta

OUTPUT_FILE = 'competitive_swings_2023_2026.csv'

DATE_RANGES = [
    ('2023-07-01', '2023-10-01'),
    ('2024-03-28', '2024-06-30'),
    ('2024-07-01', '2024-10-01'),
    ('2025-03-27', '2025-06-30'),
    ('2025-07-01', '2025-10-01'),
    ('2026-03-26', '2026-06-30'),
    ('2026-07-01', '2026-10-01'),
]

SWING_DESCRIPTIONS = [
    'hit_into_play', 'swinging_strike', 'swinging_strike_blocked',
    'foul', 'foul_tip', 'foul_bunt', 'missed_bunt', 'bunt_foul_tip',
    'hit_into_play_no_out', 'hit_into_play_score',
]

CORE_COLS = [
    'game_date', 'batter', 'pitcher', 'player_name', 'stand', 'p_throws',
    'pitch_type', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z',
    'plate_x', 'plate_z', 'balls', 'strikes', 'description', 'events',
    'launch_speed', 'launch_angle', 'estimated_woba_using_speedangle',
    'bat_speed', 'swing_length', 'attack_angle', 'attack_direction', 'swing_path_tilt',
    'sz_top', 'sz_bot',
]

def pull_chunk(start, end, chunk_days=14):
    frames = []
    current = datetime.strptime(start, '%Y-%m-%d')
    end_dt = datetime.strptime(end, '%Y-%m-%d')
    today = datetime.today()
    if end_dt > today:
        end_dt = today - timedelta(days=1)
        print(f'  Adjusted end to {end_dt.strftime("%Y-%m-%d")}')
    if current > end_dt:
        print(f'  Skipping {start} to {end} (future)')
        return pd.DataFrame()
    while current <= end_dt:
        chunk_end = min(current + timedelta(days=chunk_days - 1), end_dt)
        s, e = current.strftime('%Y-%m-%d'), chunk_end.strftime('%Y-%m-%d')
        print(f'  {s} -> {e} ...', end=' ', flush=True)
        try:
            df = statcast(start_dt=s, end_dt=e)
            if df is not None and len(df) > 0:
                frames.append(df)
                print(f'({len(df):,})')
            else: print('(no data)')
        except Exception as exc:
            print(f'ERROR: {exc}'); time.sleep(10)
        current = chunk_end + timedelta(days=1); time.sleep(2)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

all_frames = []
for start, end in DATE_RANGES:
    print(f'\n--- {start} to {end} ---')
    raw = pull_chunk(start, end)
    if raw.empty: continue
    swings = raw[raw['description'].isin(SWING_DESCRIPTIONS)].copy()
    if 'bat_speed' in swings.columns:
        swings = swings.dropna(subset=['bat_speed']).copy()
        thresholds = swings.groupby('batter')['bat_speed'].quantile(0.10).rename('p10')
        swings = swings.merge(thresholds, on='batter', how='left')
        mask = (swings['bat_speed'] >= swings['p10']) | (
            (swings['bat_speed'] >= 60) & (swings['launch_speed'] >= 90))
        swings = swings[mask].drop(columns=['p10']).copy()
    print(f'  Competitive swings: {len(swings):,}')
    all_frames.append(swings)

if all_frames:
    df = pd.concat(all_frames, ignore_index=True).drop_duplicates()
    available = [c for c in CORE_COLS if c in df.columns]
    df = df[available].copy().sort_values(['game_date', 'batter']).reset_index(drop=True)
    df.to_csv(OUTPUT_FILE, index=False)
    years = pd.to_datetime(df['game_date']).dt.year.value_counts().sort_index()
    print(f'\nSaved {len(df):,} swings -> {OUTPUT_FILE}')
    print(f'Years: {years.to_dict()}')

## 3 — Enrich Data

All feature engineering. Context features (location, height, handedness) are used to BUILD
adjustability features but never enter the model directly.

In [ ]:
import pandas as pd
import numpy as np
import os

csv_file = None
for f in ['competitive_swings_2023_2026.csv', 'competitive_swings_2023_2025.csv']:
    if os.path.exists(f): csv_file = f; break
if csv_file is None: raise FileNotFoundError('No competitive_swings CSV found')

df = pd.read_csv(csv_file)
print(f'Loaded {len(df):,} rows from {csv_file}')
print(f'Years: {pd.to_datetime(df["game_date"]).dt.year.value_counts().sort_index().to_dict()}')
STARTING_ROWS = len(df)

CONTACT = ['hit_into_play', 'foul', 'hit_into_play_no_out', 'hit_into_play_score', 'foul_bunt']
df['made_contact'] = df['description'].isin(CONTACT).astype(int)
print(f'Contact rate: {df["made_contact"].mean():.1%}')

# ═══════════════════════════════════════════════════════════════
# CONTEXT — used to build features, NOT model inputs
# ═══════════════════════════════════════════════════════════════

if all(c in df.columns for c in ['plate_x', 'plate_z']):
    df['location_difficulty'] = np.sqrt(df['plate_x']**2 + (df['plate_z'] - 2.5)**2)

if all(c in df.columns for c in ['sz_top', 'sz_bot']):
    df['sz_height'] = df['sz_top'] - df['sz_bot']
    df['pitch_height_norm'] = (df['plate_z'] - df['sz_bot']) / df['sz_height'].replace(0, np.nan)
else:
    df['sz_height'] = 1.85
    df['pitch_height_norm'] = (df['plate_z'] - 1.5) / 1.85

# ═══════════════════════════════════════════════════════════════
# SPEED CONTEXT FEATURES
# ═══════════════════════════════════════════════════════════════

# Speed over expected for that plate_x location
if all(c in df.columns for c in ['bat_speed', 'plate_x']):
    df['plate_x_bin'] = pd.cut(df['plate_x'], bins=20, labels=False)
    df['speed_over_expected'] = (df['bat_speed'] - df.groupby('plate_x_bin')['bat_speed'].transform('mean')).fillna(0)
    df.drop(columns=['plate_x_bin'], inplace=True)
    print(f'Speed over expected: mean={df["speed_over_expected"].mean():.3f}')

# Speed vs location difficulty
if all(c in df.columns for c in ['bat_speed', 'location_difficulty']):
    df['speed_vs_location'] = df['bat_speed'] * (1 + 0.3 * df['location_difficulty'])

# ═══════════════════════════════════════════════════════════════
# SWING ADJUSTABILITY — distance from empirical optimals
# ═══════════════════════════════════════════════════════════════

# ── Attack Angle vs Optimal (INCREASES with pitch height) ──
# Empirical: low=10°, mid=16°, high=27°
if all(c in df.columns for c in ['attack_angle', 'plate_z']):
    pz = df['plate_z'].clip(1.0, 4.0)
    df['optimal_aa'] = (3 * (pz - 1.0)**2 + 7).clip(5, 30)
    df['aa_vs_optimal'] = -np.abs(df['attack_angle'] - df['optimal_aa'])
    df.drop(columns=['optimal_aa'], inplace=True)
    print(f'AA vs optimal: mean={df["aa_vs_optimal"].mean():.2f}')

# ── AA Adjustment (height-normalized, player-aware) ──
if all(c in df.columns for c in ['attack_angle', 'pitch_height_norm']):
    phn = df['pitch_height_norm'].clip(0, 1.5)
    opt_aa_n = (3 * (phn * 1.85)**2 + 7).clip(5, 30)
    df['aa_adjustment'] = -np.abs(df['attack_angle'] - opt_aa_n)
    print(f'AA adjustment (height-aware): mean={df["aa_adjustment"].mean():.2f}')

# ── Tilt for Height (DECREASES with pitch height) ──
# Empirical: low=47°, mid=35°, high=29°
if all(c in df.columns for c in ['swing_path_tilt', 'plate_z']):
    pz = df['plate_z'].clip(1.0, 4.0)
    opt_tilt = (58 - 10 * pz).clip(20, 48)
    df['tilt_for_height'] = -np.abs(df['swing_path_tilt'] - opt_tilt)
    print(f'Tilt for height: mean={df["tilt_for_height"].mean():.2f}')

# ── Direction from Optimal (handedness-aware) ──
# Empirical: pull inside, oppo outside. Optimal ≈ 20 × batter_px
if all(c in df.columns for c in ['attack_direction', 'plate_x', 'stand']):
    batter_px = np.where(df['stand'] == 'R', -df['plate_x'], df['plate_x'])
    batter_ad = np.where(df['stand'] == 'R', -df['attack_direction'], df['attack_direction'])
    optimal_dir = (20 * batter_px).clip(-18, 18)
    df['direction_from_optimal'] = -np.abs(batter_ad - optimal_dir)
    print(f'Direction from optimal: mean={df["direction_from_optimal"].mean():.2f}')

# ── Length for Location ──
# Shorter swings on outside, longer on inside
if all(c in df.columns for c in ['swing_length', 'plate_x']):
    expected_len = 7.3 + 0.3 * df['plate_x']
    df['length_for_location'] = -(df['swing_length'] - expected_len)
    print(f'Length for location: mean={df["length_for_location"].mean():.3f}')

# ═══════════════════════════════════════════════════════════════
# EFFORT LEVEL — bat speed relative to personal ceiling
# ═══════════════════════════════════════════════════════════════

if 'bat_speed' in df.columns:
    df['p90_speed'] = df.groupby('batter')['bat_speed'].transform('quantile', 0.90)
    df['effort_level'] = df['bat_speed'] / df['p90_speed'].replace(0, np.nan)
    print(f'Effort level: mean={df["effort_level"].mean():.3f}')

# ═══════════════════════════════════════════════════════════════
# HARD SWING CONTACT RATE — player-level trait
# When you swing 90th+ percentile, do you still make contact?
# ═══════════════════════════════════════════════════════════════

if all(c in df.columns for c in ['bat_speed', 'made_contact', 'p90_speed']):
    is_hard = df['bat_speed'] >= df['p90_speed']
    hard_contact = df[is_hard].groupby('batter')['made_contact'].mean().rename('hard_swing_contact')
    pre = len(df)
    df = df.merge(hard_contact, on='batter', how='left')
    assert len(df) == pre
    print(f'Hard swing contact rate: mean={df["hard_swing_contact"].mean():.3f}')

# ═══════════════════════════════════════════════════════════════
# ADAPTABILITY TRAITS — player-level
# Do they systematically adjust mechanics to pitch location?
# ═══════════════════════════════════════════════════════════════

# ── AA Adaptability: correlation of attack_angle with plate_z ──
# Positive = adjusts AA up on high pitches (empirically optimal)
if all(c in df.columns for c in ['attack_angle', 'plate_z']):
    def calc_aa_adapt(g):
        if len(g) < 50: return np.nan
        return g['attack_angle'].corr(g['plate_z'])
    aa_adapt = df.groupby('batter').apply(calc_aa_adapt).rename('aa_adaptability')
    pre = len(df)
    df = df.merge(aa_adapt, on='batter', how='left')
    assert len(df) == pre
    print(f'AA adaptability: mean={df["aa_adaptability"].mean():.3f} (positive = adjusts correctly)')

# ── Direction Adaptability: correlation of batter_ad with batter_px ──
# Positive = pulls inside, goes oppo outside (empirically optimal)
if all(c in df.columns for c in ['attack_direction', 'plate_x', 'stand']):
    def calc_dir_adapt(g):
        if len(g) < 50: return np.nan
        bpx = np.where(g['stand'] == 'R', -g['plate_x'], g['plate_x'])
        bad = np.where(g['stand'] == 'R', -g['attack_direction'], g['attack_direction'])
        bpx_s = pd.Series(bpx, index=g.index)
        bad_s = pd.Series(bad, index=g.index)
        return bad_s.corr(bpx_s)
    dir_adapt = df.groupby('batter').apply(calc_dir_adapt).rename('dir_adaptability')
    pre = len(df)
    df = df.merge(dir_adapt, on='batter', how='left')
    assert len(df) == pre
    print(f'Dir adaptability: mean={df["dir_adaptability"].mean():.3f} (positive = adjusts correctly)')

# ═══════════════════════════════════════════════════════════════
# TARGET: xwOBAcon
# ═══════════════════════════════════════════════════════════════

contact_mask = df['launch_speed'].notna() & df['launch_angle'].notna()
if 'estimated_woba_using_speedangle' in df.columns:
    df['xwOBAcon'] = np.where(contact_mask, df['estimated_woba_using_speedangle'], np.nan)
    print(f'\nxwOBAcon: {df["xwOBAcon"].notna().sum():,} values, mean={df["xwOBAcon"].mean():.3f}')

# ═══════════════════════════════════════════════════════════════
# BATTER NAMES
# ═══════════════════════════════════════════════════════════════

if 'player_name' in df.columns:
    df = df.rename(columns={'player_name': 'pitcher_name'})

savant_file = 'stats__39_.csv'
if os.path.exists(savant_file):
    ps = pd.read_csv(savant_file)
    ps = ps.loc[:, ~ps.columns.duplicated()]
    for col in ['xslg', 'xwoba', 'xwobacon']:
        if col in ps.columns: ps[col] = pd.to_numeric(ps[col].astype(str).str.strip('"'), errors='coerce')
    nl = ps.drop_duplicates('player_id')[['player_id', 'last_name, first_name']].copy()
    nl.columns = ['batter', 'batter_name']
    nl['batter'] = nl['batter'].astype(int)
    pa = ps.groupby('player_id').agg(player_xSLG=('xslg','mean'), player_xwOBA=('xwoba','mean'), player_xwOBAcon=('xwobacon','mean')).reset_index()
    pa['player_id'] = pa['player_id'].astype(int)
    df['batter'] = df['batter'].astype(int)
    pre = len(df); df = df.merge(nl, on='batter', how='left'); assert len(df) == pre
    pre = len(df); df = df.merge(pa, left_on='batter', right_on='player_id', how='left')
    if 'player_id' in df.columns: df.drop(columns=['player_id'], inplace=True)
    assert len(df) == pre
    print(f'Batter names: {df["batter_name"].notna().sum():,} / {len(df):,}')
else:
    print(f'{savant_file} not found — using pybaseball')
    df['batter'] = df['batter'].astype(int)
    try:
        from pybaseball import playerid_reverse_lookup
        all_ids = df['batter'].unique()
        for i in range(0, len(all_ids), 100):
            batch = all_ids[i:i+100]
            try:
                lk = playerid_reverse_lookup(batch, key_type='mlbam')
                for _, r in lk.iterrows(): df.loc[df['batter']==r['key_mlbam'],'batter_name'] = r['name_last']+', '+r['name_first']
            except: pass
        print(f'Batter names: {df["batter_name"].notna().sum():,} / {len(df):,}')
    except: print('Name lookup failed')

# Fill any remaining missing names
missing_ids = df[df.get('batter_name', pd.Series(dtype=str)).isna()]['batter'].unique()
if len(missing_ids) > 0:
    try:
        from pybaseball import playerid_reverse_lookup
        lk = playerid_reverse_lookup(missing_ids, key_type='mlbam')
        for _, r in lk.iterrows(): df.loc[df['batter']==r['key_mlbam'],'batter_name'] = r['name_last']+', '+r['name_first']
    except: pass

assert len(df) == STARTING_ROWS
df.to_csv('competitive_swings_enriched.csv', index=False)
years = pd.to_datetime(df['game_date']).dt.year.value_counts().sort_index()
print(f'\nSaved {len(df):,} rows  |  Years: {years.to_dict()}  |  Columns: {len(df.columns)}')

# Feature summary
print('\n── v9 Feature Summary ──')
for f in ['bat_speed', 'swing_length', 'speed_over_expected', 'speed_vs_location',
          'aa_vs_optimal', 'aa_adjustment', 'tilt_for_height', 'direction_from_optimal',
          'length_for_location', 'effort_level', 'hard_swing_contact', 'aa_adaptability', 'dir_adaptability']:
    if f in df.columns:
        print(f'  {f:25s}  n={df[f].notna().sum():>10,}  mean={df[f].mean():+.4f}')
    else:
        print(f'  {f:25s}  MISSING')

## 4 — Train Unified Model

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

df = pd.read_csv('competitive_swings_enriched.csv')
print(f'Loaded {len(df):,} rows')

FEATURES = [
    # Raw swing mechanics
    'bat_speed',
    'swing_length',
    # Speed context
    'speed_over_expected',
    'speed_vs_location',
    # Swing adjustability (distance from empirical optimal)
    'aa_vs_optimal',
    'aa_adjustment',
    'tilt_for_height',
    'direction_from_optimal',
    'length_for_location',
    # Effort
    'effort_level',
    # Player traits
    'hard_swing_contact',
    'aa_adaptability',
    'dir_adaptability',
]

contact_df = df[df['xwOBAcon'].notna() & df['bat_speed'].notna()].copy()
print(f'Contact swings with xwOBAcon: {len(contact_df):,}')

available = [f for f in FEATURES if f in contact_df.columns]
missing = [f for f in FEATURES if f not in contact_df.columns]
if missing: print(f'Missing: {missing}')
print(f'Using {len(available)} features')

X = contact_df[available].copy()
y = contact_df['xwOBAcon'].copy()

fill_medians = {}
for col in X.columns:
    med = X[col].median(); fill_medians[col] = med
    n = X[col].isna().sum()
    if n > 0: print(f'  Filling {n:,} missing in {col}'); X[col] = X[col].fillna(med)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

model = XGBRegressor(
    n_estimators=800, max_depth=7, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=50,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='rmse', early_stopping_rounds=50,
    random_state=42, n_jobs=-1)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
r = np.corrcoef(y_test, y_pred)[0, 1]
mae = mean_absolute_error(y_test, y_pred)

print(f'\n── Unified Model Results ──')
print(f'  R2  = {r2:.4f}')
print(f'  r   = {r:.4f}')
print(f'  MAE = {mae:.4f}')
print(f'  Best iteration: {model.best_iteration}')

print(f'\n  Feature Importances:')
for feat, imp in sorted(zip(available, model.feature_importances_), key=lambda x: -x[1]):
    bar = chr(9608) * int(imp * 60)
    print(f'    {feat:25s} {imp:.4f}  {bar}')

## 5 — Compute iSwing+, Validate, Export

In [ ]:
import joblib, json

joblib.dump(model, 'iswing_model.pkl')
joblib.dump(scaler, 'iswing_scaler.pkl')

all_df = df.dropna(subset=['bat_speed']).copy()
name_col = 'batter_name' if 'batter_name' in all_df.columns else 'pitcher_name'
all_df = all_df[all_df[name_col].notna()].copy()
all_df['year'] = pd.to_datetime(all_df['game_date'], errors='coerce').dt.year
print(f'Scoring {len(all_df):,} swings...')

X_all = all_df[available].copy()
for col in X_all.columns:
    X_all[col] = X_all[col].fillna(fill_medians.get(col, X_all[col].median()))
all_df['pred_wOBAcon'] = model.predict(
    pd.DataFrame(scaler.transform(X_all), columns=X_all.columns, index=X_all.index))
all_df['raw_value'] = all_df['pred_wOBAcon'].clip(lower=0.001)

# ── Player-level iSwing+ ──
player_sp = all_df.groupby(name_col).agg(
    raw_value=('raw_value', 'mean'), pred_wOBAcon=('pred_wOBAcon', 'mean'),
    n_swings=('raw_value', 'count'), bat_speed=('bat_speed', 'mean'),
    swing_length=('swing_length', 'mean'), actual_contact=('made_contact', 'mean'),
).reset_index()
player_sp = player_sp[(player_sp['n_swings'] >= 150) & player_sp[name_col].notna()].copy()

player_sp['log_raw'] = np.log(player_sp['raw_value'].clip(lower=1e-10))
log_mean = player_sp['log_raw'].mean()
log_std = player_sp['log_raw'].std()
player_sp['iSwing+'] = (100 + 15 * (player_sp['log_raw'] - log_mean) / log_std).round(0).astype(int)

print(f'Players: {len(player_sp)}  |  Mean={player_sp["iSwing+"].mean():.1f}  Std={player_sp["iSwing+"].std():.1f}')

# Per-year
years = sorted(all_df['year'].dropna().astype(int).unique())
for yr in years:
    ya = all_df[all_df['year']==yr].groupby(name_col)['raw_value'].agg(['mean','count']).reset_index()
    ya = ya[ya['count'] >= 25]
    if len(ya) > 0:
        ya['lr'] = np.log(ya['mean'].clip(lower=1e-10))
        ya[str(yr)] = (100 + 15*(ya['lr']-ya['lr'].mean())/ya['lr'].std()).round(0).astype(int)
        player_sp = player_sp.merge(ya[[name_col, str(yr)]], on=name_col, how='left')
    print(f'  {yr}: {len(ya)} players')

# Validation
for tc in ['player_xSLG', 'player_xwOBA', 'player_xwOBAcon']:
    if tc in all_df.columns:
        pt = all_df.groupby(name_col)[tc].mean().reset_index()
        player_sp = player_sp.merge(pt, on=name_col, how='left')

print(f'\n{"="*60}')
print('VALIDATION:')
for t in ['player_xSLG', 'player_xwOBA', 'player_xwOBAcon']:
    if t in player_sp.columns:
        v = player_sp[['iSwing+', t]].dropna(); r = v['iSwing+'].corr(v[t])
        print(f'  iSwing+ vs {t:20s}  r={r:+.4f}  R2={r**2:.4f}')

def flip_name(n):
    if ',' in str(n):
        parts = n.split(',', 1)
        return f'{parts[1].strip()} {parts[0].strip()}'
    return n

for label, sub in [('TOP 25', player_sp.nlargest(25,'iSwing+')), ('BOTTOM 25', player_sp.nsmallest(25,'iSwing+'))]:
    print(f'\n  {label}:')
    for _, row in sub.iterrows():
        dn = flip_name(row[name_col])
        x = ''
        if 'player_xSLG' in row.index and pd.notna(row.get('player_xSLG')): x += f'  xSLG={row["player_xSLG"]:.3f}'
        print(f'    {dn:25s}  iSwing+={row["iSwing+"]:>4d}  wOBAcon={row["pred_wOBAcon"]:.3f}  Spd={row["bat_speed"]:.1f}{x}')

# Export
year_cols = sorted([c for c in player_sp.columns if c.isdigit() and len(c)==4])
export = player_sp[[name_col,'iSwing+']+year_cols+['pred_wOBAcon','actual_contact','bat_speed','swing_length','n_swings']].copy()
export.rename(columns={name_col:'batter_name', 'pred_wOBAcon':'quality'}, inplace=True)
export['batter_name'] = export['batter_name'].apply(flip_name)
export['quality'] = export['quality'].round(3)
export['actual_contact'] = (export['actual_contact']*100).round(1)
export['bat_speed'] = export['bat_speed'].round(1)
export['swing_length'] = export['swing_length'].round(2)
for yc in year_cols: export[yc] = export[yc].astype('Int64')
export.sort_values('iSwing+', ascending=False).reset_index(drop=True).to_csv('iSwing_leaderboard.csv', index=False)
print(f'\nExported {len(export)} players -> iSwing_leaderboard.csv')

config = {'features': list(available), 'log_mean': float(log_mean), 'log_std': float(log_std),
    'target_mean': 100, 'target_std': 15,
    'feature_medians': {k: float(v) for k,v in fill_medians.items()}}
with open('iswing_config.json', 'w') as f: json.dump(config, f, indent=2)
print('Config saved.')

## 6 — Player Decomposition (Waterfall)

Each feature's contribution in iSwing+ points, starting from 100.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, joblib, json

df = pd.read_csv('competitive_swings_enriched.csv')
model = joblib.load('iswing_model.pkl')
scaler = joblib.load('iswing_scaler.pkl')
with open('iswing_config.json') as f: config = json.load(f)
available = config['features']
fill_medians = config['feature_medians']

all_df = df.dropna(subset=['bat_speed']).copy()
name_col = 'batter_name' if 'batter_name' in all_df.columns else 'pitcher_name'
all_df['year'] = pd.to_datetime(all_df['game_date'], errors='coerce').dt.year
all_df = all_df[all_df[name_col].notna()].copy()
print(f'Loaded {len(all_df):,} swings')

# League baselines
med = {f: all_df[f].median() for f in available if f in all_df.columns}
std = {f: max(all_df[f].std(), 0.001) for f in available if f in all_df.columns}
imp = dict(zip(available, model.feature_importances_))

# Player averages
feat_cols = [f for f in available if f in all_df.columns]
player_avgs = all_df.groupby(name_col)[feat_cols].mean()
player_n = all_df.groupby(name_col).size()
player_avgs = player_avgs[player_n >= 150]

# Per-year averages
yr_avgs = {}
for yr in sorted(all_df['year'].dropna().astype(int).unique()):
    yr_df = all_df[all_df['year'] == yr]
    ya = yr_df.groupby(name_col)[feat_cols].mean()
    yn = yr_df.groupby(name_col).size()
    yr_avgs[yr] = ya[yn >= 25]
    print(f'  {yr}: {len(yr_avgs[yr])} players')

# Rebuild player_sp and per-year iSwing+
tmp = all_df.copy()
for c in available:
    if c in tmp.columns: tmp[c] = tmp[c].fillna(med.get(c, 0))
tmp['pred'] = model.predict(pd.DataFrame(scaler.transform(tmp[available]), columns=available))
tmp['raw'] = tmp['pred'].clip(lower=0.001)
player_sp = tmp.groupby(name_col).agg(raw_value=('raw','mean'), n_swings=('raw','count')).reset_index()
player_sp = player_sp[player_sp['n_swings'] >= 150].copy()
player_sp['log_raw'] = np.log(player_sp['raw_value'].clip(lower=1e-10))
lm, ls = player_sp['log_raw'].mean(), player_sp['log_raw'].std()
player_sp['iSwing+'] = (100 + 15*(player_sp['log_raw']-lm)/ls).round(0).astype(int)

yr_raw = {}
for yr in sorted(all_df['year'].dropna().astype(int).unique()):
    ya = tmp[tmp['year']==yr].groupby(name_col)['raw'].agg(['mean','count']).reset_index()
    ya = ya[ya['count'] >= 25]
    if len(ya) > 0:
        ya['lr'] = np.log(ya['mean'].clip(lower=1e-10))
        ya['iswing_yr'] = (100 + 15*(ya['lr']-ya['lr'].mean())/ya['lr'].std()).round(0).astype(int)
        yr_raw[yr] = ya.set_index(name_col)['iswing_yr']
        player_sp = player_sp.merge(ya[[name_col, 'iswing_yr']].rename(columns={'iswing_yr': str(yr)}), on=name_col, how='left')

print('Done!\n')

def flip_name(n):
    if ',' in str(n):
        parts = n.split(',', 1)
        return f'{parts[1].strip()} {parts[0].strip()}'
    return n

def compute_waterfall(player_avgs_row, target_iswing):
    total_shift = target_iswing - 100
    raw_contribs = {}
    for f in available:
        if f not in player_avgs_row.index: continue
        z = (player_avgs_row[f] - med.get(f, 0)) / std.get(f, 1)
        raw_contribs[f] = imp.get(f, 0) * z
    raw_total = sum(raw_contribs.values())
    if abs(raw_total) < 0.001:
        return {f: total_shift / max(len(raw_contribs), 1) for f in raw_contribs}
    scale = total_shift / raw_total
    return {f: v * scale for f, v in raw_contribs.items()}


def decompose_player(player_name):
    player_years = [yr for yr in yr_avgs if player_name in yr_avgs[yr].index]
    if not player_years:
        print(f'"{player_name}" not found.')
        last = player_name.split(',')[0].strip()
        matches = [n for n in player_avgs.index if last.lower() in n.lower()]
        if matches: print(f'  Did you mean: {matches[:5]}')
        return

    ps_row = player_sp[player_sp[name_col] == player_name]
    iswing_overall = ps_row['iSwing+'].values[0] if len(ps_row) > 0 else 100
    display_name = flip_name(player_name)

    n_cols = len(player_years)
    fig, axes = plt.subplots(1, n_cols, figsize=(8*n_cols, max(7, len(available)*0.55)))
    if n_cols == 1: axes = [axes]

    print(f'\n{"="*70}')
    print(f'{display_name}  (Overall iSwing+ = {iswing_overall})')
    print(f'{"="*70}')

    for i, yr in enumerate(player_years):
        pavg = yr_avgs[yr].loc[player_name]
        yr_iswing = 100
        if yr in yr_raw and player_name in yr_raw[yr].index:
            yr_iswing = int(yr_raw[yr][player_name])

        contribs = compute_waterfall(pavg, yr_iswing)
        sorted_feats = sorted(contribs.items(), key=lambda x: abs(x[1]), reverse=True)
        feat_names = [f[0] for f in sorted_feats]
        feat_vals = [f[1] for f in sorted_feats]

        running = [100]
        for v in feat_vals: running.append(running[-1] + v)

        colors = ['#2563eb' if v >= 0 else '#dc2626' for v in feat_vals]
        lefts = [min(running[j], running[j+1]) for j in range(len(feat_vals))]

        axes[i].barh(range(len(feat_names)), [abs(v) for v in feat_vals],
                     left=lefts, color=colors, height=0.7)
        for j, (f, v) in enumerate(zip(feat_names, feat_vals)):
            end = running[j+1]
            axes[i].text(end + (0.3 if v >= 0 else -0.3), j,
                        f'{v:+.1f}', va='center', ha='left' if v >= 0 else 'right',
                        fontsize=8, fontweight='bold', color='gray')
        axes[i].set_yticks(range(len(feat_names)))
        axes[i].set_yticklabels(feat_names, fontsize=9)
        axes[i].axvline(100, color='gray', linewidth=1, linestyle='--', alpha=0.7)
        axes[i].axvline(yr_iswing, color='black', linewidth=1.5, linestyle='-', alpha=0.4)
        axes[i].text(100, len(feat_names)+0.3, 'Avg (100)', ha='center', fontsize=8, color='gray')
        axes[i].text(yr_iswing, -0.7, f'iSwing+ = {yr_iswing}', ha='center', fontsize=10, fontweight='bold', color='black')
        axes[i].set_xlabel('iSwing+ Points')
        axes[i].set_title(f'{yr} (iSwing+ = {yr_iswing})', fontweight='bold', fontsize=13)
        axes[i].invert_yaxis()

        print(f'\n  ── {yr} (iSwing+ = {yr_iswing}) ──')
        print(f'  {"Feature":25s}  {"Shift":>7s}  {"Running":>7s}')
        print(f'  {"-"*45}')
        print(f'  {"START":25s}  {"":>7s}  {"100":>7s}')
        run = 100
        for f, v in zip(feat_names, feat_vals):
            run += v
            print(f'  {f:25s}  {v:+7.1f}  {run:7.1f}')
        print(f'  {"FINAL":25s}  {"":>7s}  {yr_iswing:>7d}')

    fig.suptitle(f'{display_name} — iSwing+ Waterfall Decomposition (Overall = {iswing_overall})',
                 fontweight='bold', fontsize=16, y=1.02)
    plt.tight_layout()
    safe = display_name.replace(' ', '_')
    fname = f'{safe}_iSwing_Decomp.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved {fname}')


# Run top 5
for name in player_sp.nlargest(5, 'iSwing+')[name_col].values:
    decompose_player(name)

# Save for Streamlit
for yr in yr_avgs:
    rows = []
    for player in yr_avgs[yr].index:
        yr_iswing = 100
        if yr in yr_raw and player in yr_raw[yr].index: yr_iswing = int(yr_raw[yr][player])
        contribs = compute_waterfall(yr_avgs[yr].loc[player], yr_iswing)
        contribs[name_col] = player
        contribs['iSwing+'] = yr_iswing
        rows.append(contribs)
    pd.DataFrame(rows).to_csv(f'player_decomp_{yr}.csv', index=False)
print('\nCSVs saved.')
print('\n  decompose_player("Judge, Aaron")')
print('  decompose_player("Soto, Juan")')

## 7 — Write Streamlit App

In [ ]:
%%writefile app.py
import json, joblib, glob, io
import numpy as np, pandas as pd
import streamlit as st
import plotly.express as px
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

st.set_page_config(page_title='iSwing+', page_icon='baseball', layout='wide')

@st.cache_resource
def load_model():
    try:
        m = joblib.load('iswing_model.pkl'); s = joblib.load('iswing_scaler.pkl')
        with open('iswing_config.json') as f: c = json.load(f)
        return m, s, c
    except: return None, None, None

@st.cache_data
def load_data():
    try:
        df = pd.read_csv('competitive_swings_enriched.csv')
        for dc in ['player_name','pitcher_name']:
            if dc in df.columns and 'batter_name' in df.columns: df = df.drop(columns=[dc])
        return df
    except: return None

@st.cache_data
def load_decomp():
    yearly = {}
    for f in sorted(glob.glob('player_decomp_*.csv')):
        name = f.replace('player_decomp_','').replace('.csv','')
        if name == 'overall': continue
        try: yearly[int(name)] = pd.read_csv(f)
        except: pass
    return yearly

def flip_name(n):
    if ',' in str(n):
        p = n.split(',',1); return f'{p[1].strip()} {p[0].strip()}'
    return n

def color_for(v):
    if v>=130: return '#dc2626'
    if v>=115: return '#ea580c'
    if v>=100: return '#2563eb'
    if v>=85: return '#6b7280'
    return '#9ca3af'

def raw_to_iswing(s):
    lr = np.log(s.clip(lower=1e-10))
    return (100+15*(lr-lr.mean())/lr.std()).round(0).astype(int)

def make_waterfall_png(player_name, yearly, nc):
    display_name = flip_name(player_name)
    player_years = sorted([yr for yr in yearly if player_name in yearly[yr][nc].values])
    if not player_years: return None, None
    feat_cols = [c for c in yearly[player_years[0]].columns if c not in [nc,'iSwing+']]
    n_cols = len(player_years)
    fig, axes = plt.subplots(1, n_cols, figsize=(8*n_cols, max(7, len(feat_cols)*0.55)))
    if n_cols == 1: axes = [axes]
    for i, yr in enumerate(player_years):
        row = yearly[yr][yearly[yr][nc] == player_name].iloc[0]
        yr_iswing = int(row.get('iSwing+', 100))
        contribs = {f: row[f] for f in feat_cols if f in row.index and pd.notna(row[f])}
        sorted_feats = sorted(contribs.items(), key=lambda x: abs(x[1]), reverse=True)
        feat_names = [f[0] for f in sorted_feats]
        feat_vals = [f[1] for f in sorted_feats]
        running = [100]
        for v in feat_vals: running.append(running[-1]+v)
        colors = ['#2563eb' if v>=0 else '#dc2626' for v in feat_vals]
        lefts = [min(running[j], running[j+1]) for j in range(len(feat_vals))]
        axes[i].barh(range(len(feat_names)), [abs(v) for v in feat_vals], left=lefts, color=colors, height=0.7)
        for j, (f, v) in enumerate(zip(feat_names, feat_vals)):
            end = running[j+1]
            axes[i].text(end+(0.3 if v>=0 else -0.3), j, f'{v:+.1f}', va='center',
                        ha='left' if v>=0 else 'right', fontsize=8, fontweight='bold', color='gray')
        axes[i].set_yticks(range(len(feat_names)))
        axes[i].set_yticklabels(feat_names, fontsize=9)
        axes[i].axvline(100, color='gray', linewidth=1, linestyle='--', alpha=0.7)
        axes[i].axvline(yr_iswing, color='black', linewidth=1.5, linestyle='-', alpha=0.4)
        axes[i].text(100, len(feat_names)+0.3, 'Avg (100)', ha='center', fontsize=8, color='gray')
        axes[i].text(yr_iswing, -0.7, f'iSwing+ = {yr_iswing}', ha='center', fontsize=10, fontweight='bold', color='black')
        axes[i].set_xlabel('iSwing+ Points')
        axes[i].set_title(f'{yr} (iSwing+ = {yr_iswing})', fontweight='bold', fontsize=13)
        axes[i].invert_yaxis()
    fig.suptitle(f'{display_name} — iSwing+ Waterfall Decomposition', fontweight='bold', fontsize=16, y=1.02)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight'); buf.seek(0)
    plt.close(fig)
    return buf, display_name

def main():
    m, s, config = load_model()
    df = load_data()
    nc = 'batter_name' if df is not None and 'batter_name' in df.columns else 'pitcher_name'
    st.title('iSwing+ v9')
    st.caption('Swing quality + adjustability | Single model | mean=100, std=15')
    if m is None: st.warning('Model files not found.')

    page = st.sidebar.radio('Navigation', ['Calculator','Leaderboard','Decompose','Compare','About'])

    if page == 'Calculator':
        st.header('iSwing+ Calculator')
        c1,c2,c3 = st.columns(3)
        with c1:
            bs = st.slider('Bat Speed (mph)',55.0,90.0,72.0,0.5)
            sl = st.slider('Swing Length (ft)',4.0,10.0,7.3,0.1)
        with c2:
            aa = st.slider('Attack Angle (deg)',-15.0,40.0,12.0,0.5)
            spt = st.slider('Swing Path Tilt (deg)',10.0,55.0,32.0,0.5)
        with c3:
            px_ = st.slider('Plate X (ft)',-1.5,1.5,0.0,0.05)
            pz = st.slider('Plate Z (ft)',1.0,4.5,2.5,0.05)
            hand = st.selectbox('Batter Hand', ['R','L'])

        loc_diff = np.sqrt(px_**2+(pz-2.5)**2)
        pz_c = np.clip(pz,1.0,4.0)
        opt_aa = np.clip(3*(pz_c-1.0)**2+7, 5, 30)
        phn = (pz-1.5)/1.85
        opt_aa_n = np.clip(3*(np.clip(phn,0,1.5)*1.85)**2+7, 5, 30)
        opt_tilt = np.clip(58-10*pz_c, 20, 48)
        exp_len = 7.3+0.3*px_
        bpx = -px_ if hand=='R' else px_
        ad_guess = -5.0
        bad = -ad_guess if hand=='R' else ad_guess
        opt_dir = np.clip(20*bpx, -18, 18)

        feat = {
            'bat_speed':bs, 'swing_length':sl,
            'speed_over_expected':bs-72.0, 'speed_vs_location':bs*(1+0.3*loc_diff),
            'aa_vs_optimal':-abs(aa-opt_aa), 'aa_adjustment':-abs(aa-opt_aa_n),
            'tilt_for_height':-abs(spt-opt_tilt),
            'direction_from_optimal':-abs(bad-opt_dir),
            'length_for_location':-(sl-exp_len),
            'effort_level':bs/75.0,
            'hard_swing_contact':0.65, 'aa_adaptability':0.15, 'dir_adaptability':0.10,
        }
        st.divider()
        if m:
            fs = config['features']
            row = {n: feat.get(n, config['feature_medians'].get(n,0)) for n in fs}
            inp = pd.DataFrame([row])
            pred = max(float(m.predict(pd.DataFrame(s.transform(inp),columns=fs))[0]), 0.001)
            lr = np.log(pred)
            sp = int(round(config['target_mean']+config['target_std']*(lr-config['log_mean'])/config['log_std']))
        else: sp, pred = 100, 0.35
        col = color_for(sp)
        lbl = 'Elite' if sp>=130 else 'Above Avg' if sp>=115 else 'Average' if sp>=100 else 'Below Avg' if sp>=85 else 'Poor'
        _,mid,_ = st.columns([1,2,1])
        with mid:
            st.markdown(f"<h1 style='text-align:center;color:{col};font-size:5rem;margin:0;'>{sp}</h1>", unsafe_allow_html=True)
            st.markdown(f"<p style='text-align:center;color:{col};font-weight:bold;'>{lbl}</p>", unsafe_allow_html=True)
        st.divider()
        m1,m2,m3,m4 = st.columns(4)
        m1.metric('Pred wOBAcon',f'{pred:.3f}'); m2.metric('Bat Speed',f'{bs} mph')
        m3.metric('AA vs Optimal',f'{-abs(aa-opt_aa):+.1f}'); m4.metric('Tilt vs Optimal',f'{-abs(spt-opt_tilt):+.1f}')

    elif page == 'Leaderboard':
        st.header('iSwing+ Leaderboard')
        if df is None or m is None: st.warning('Data/model required.'); return
        sdf = df.dropna(subset=['bat_speed']).copy()
        sdf = sdf[sdf[nc].notna()].copy()
        if 'game_date' in sdf.columns: sdf['year']=pd.to_datetime(sdf['game_date'],errors='coerce').dt.year
        X = sdf[[f for f in config['features'] if f in sdf.columns]].copy()
        for c in X.columns: X[c]=X[c].fillna(X[c].median())
        sdf['pred']=np.maximum(m.predict(pd.DataFrame(s.transform(X),columns=X.columns)),0.001)
        sdf['raw_value']=sdf['pred']
        mn = st.slider('Min swings',50,500,150,25)
        ps = sdf.groupby(nc).agg(raw_value=('raw_value','mean'), quality=('pred','mean'),
            actual_contact=('made_contact','mean'), bat_speed=('bat_speed','mean'),
            n_swings=('raw_value','count')).reset_index()
        ps = ps[ps['n_swings']>=mn]
        ps['iSwing+'] = raw_to_iswing(ps['raw_value'])
        if 'year' in sdf.columns:
            for yr in sorted(sdf['year'].dropna().unique()):
                yr=int(yr)
                ya=sdf[sdf['year']==yr].groupby(nc)['raw_value'].agg(['mean','count']).reset_index()
                ya=ya[ya['count']>=25]
                if len(ya)>0: ya[str(yr)]=raw_to_iswing(ya['mean']); ps=ps.merge(ya[[nc,str(yr)]],on=nc,how='left')
        ps[nc] = ps[nc].apply(flip_name)
        ps['quality']=ps['quality'].round(3); ps['actual_contact']=(ps['actual_contact']*100).round(1)
        ps['bat_speed']=ps['bat_speed'].round(1)
        ps = ps.drop(columns=['raw_value']).sort_values('iSwing+',ascending=False).reset_index(drop=True)
        ycs = sorted([c for c in ps.columns if c.isdigit() and len(c)==4])
        for yc in ycs: ps[yc]=ps[yc].astype('Int64')
        lead = [nc,'iSwing+']+ycs+['quality','actual_contact','bat_speed','n_swings']
        ps = ps[[c for c in lead if c in ps.columns]]
        st.dataframe(ps, use_container_width=True, hide_index=True)
        fig=px.histogram(ps,x='iSwing+',nbins=30,title='iSwing+ Distribution',color_discrete_sequence=['#2563eb'])
        fig.add_vline(x=100,line_dash='dash',line_color='gray',annotation_text='Avg')
        fig.update_layout(template='plotly_white',showlegend=False)
        st.plotly_chart(fig,use_container_width=True)

    elif page == 'Decompose':
        st.header('Player Decomposition')
        st.caption('Waterfall: each feature adds or subtracts iSwing+ points from 100 (league avg)')
        yearly = load_decomp()
        if not yearly: st.warning('Run Cell 6 first.'); return
        first_yr = list(yearly.values())[0]
        ns = nc
        for try_col in ['batter_name','pitcher_name']:
            if try_col in first_yr.columns: ns = try_col; break
        all_players = set()
        for yr_df in yearly.values(): all_players.update(yr_df[ns].dropna().unique())
        display_map = {p: flip_name(p) for p in sorted(all_players)}
        display_names = sorted(display_map.values())
        reverse_map = {v: k for k, v in display_map.items()}
        sel_display = st.selectbox('Select a hitter', display_names)
        if sel_display:
            sel = reverse_map[sel_display]
            buf, dname = make_waterfall_png(sel, yearly, ns)
            if buf is None: st.warning(f'No data for {sel_display}'); return
            st.image(buf, use_container_width=True)
            st.download_button(label=f'Download {dname} decomposition',
                data=buf.getvalue(), file_name=f'{dname.replace(" ","_")}_iSwing_Decomp.png', mime='image/png')

    elif page == 'Compare':
        st.header('Compare Hitters')
        if df is None or m is None: st.warning('Need data/model.'); return
        players = sorted(df[nc].dropna().unique())
        sel = st.multiselect('Select hitters (up to 5)', [flip_name(p) for p in players], max_selections=5)
        if sel:
            rev = {flip_name(p):p for p in players}
            sel_orig = [rev[s_] for s_ in sel if s_ in rev]
            sdf = df[df[nc].isin(sel_orig)].dropna(subset=['bat_speed'])
            rows = []
            for p in sel_orig:
                pdf = sdf[sdf[nc]==p]
                if len(pdf)==0: continue
                r = {'Hitter':flip_name(p), 'Swings':len(pdf),
                     'Bat Speed':round(pdf['bat_speed'].mean(),1),
                     'Swing Length':round(pdf['swing_length'].mean(),2),
                     'Contact%':round(pdf['made_contact'].mean()*100,1)}
                if 'aa_vs_optimal' in pdf.columns: r['AA vs Opt']=round(pdf['aa_vs_optimal'].mean(),2)
                if 'direction_from_optimal' in pdf.columns: r['Dir vs Opt']=round(pdf['direction_from_optimal'].mean(),2)
                if 'effort_level' in pdf.columns: r['Effort']=round(pdf['effort_level'].mean(),3)
                rows.append(r)
            if rows: st.dataframe(pd.DataFrame(rows),use_container_width=True,hide_index=True)

    elif page == 'About':
        st.header('About iSwing+ v9')
        st.markdown("""
**iSwing+ v9** — Single unified model predicting wOBAcon from swing mechanics, adaptability, and player traits.

**Features:**
| Category | Features |
|----------|----------|
| Swing Mechanics | bat_speed, swing_length |
| Speed Context | speed_over_expected, speed_vs_location |
| Adjustability | aa_vs_optimal, aa_adjustment, tilt_for_height, direction_from_optimal, length_for_location |
| Effort | effort_level (bat speed vs personal 90th pct) |
| Player Traits | hard_swing_contact, aa_adaptability, dir_adaptability |

**Decomposition:** Waterfall chart showing exactly how many iSwing+ points each feature adds or subtracts, starting from 100 (league average).

```
iSwing+ = 100 + 15 x (ln(pred_wOBAcon) - mean) / std
```

| iSwing+ | Rating |
|---------|----------------|
| 130+    | Elite |
| 115-129 | Above Average |
| 100     | Average |
| 85-99   | Below Average |
| < 85    | Poor |
""")

if __name__ == '__main__': main()

## — ngrok auth (run once)

In [ ]:
!ngrok config add-authtoken <YOUR_NGROK_TOKEN>

## 8 — Launch Streamlit

In [ ]:
import subprocess, sys, time, os
from pyngrok import ngrok
ngrok.kill(); os.system('pkill -f streamlit 2>/dev/null'); time.sleep(3)
try:
    for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
except: pass
time.sleep(2)
proc = subprocess.Popen([sys.executable,'-m','streamlit','run','app.py',
    '--server.headless','true','--server.port','8501'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
url = ngrok.connect(8501)
print('='*50)
print(f'iSwing+ v9 is live!\n\n  {url}')
print('='*50)

In [ ]:
import pandas as pd
import numpy as np

lb = pd.read_csv('iSwing_leaderboard.csv')
df = pd.read_csv('competitive_swings_enriched.csv')

name_col = 'batter_name' if 'batter_name' in df.columns else 'pitcher_name'
df['year'] = pd.to_datetime(df['game_date'], errors='coerce').dt.year

# Actual xwOBAcon per player per year (from contact swings)
actual = df[df['xwOBAcon'].notna()].groupby([name_col, 'year']).agg(
    actual_xwobacon=('xwOBAcon', 'mean'),
    n_contact=('xwOBAcon', 'count'),
).reset_index()
actual = actual[actual['n_contact'] >= 40]

# Match names
def to_last_first(n):
    parts = str(n).strip().split(' ', 1)
    if len(parts) == 2:
        return f'{parts[1].strip()}, {parts[0].strip()}'
    return n

lb['name_raw'] = lb['batter_name'].apply(to_last_first)
year_cols = [c for c in lb.columns if c.isdigit() and len(c) == 4]

print('=' * 60)
print('CONCURRENT (same year iSwing+ vs same year xwOBAcon)')
print('=' * 60)
for yr in sorted(actual['year'].unique()):
    yr = int(yr)
    yr_col = str(yr)
    if yr_col not in lb.columns: continue
    yr_actual = actual[actual['year'] == yr]
    merged = lb[lb[yr_col].notna()].merge(yr_actual, left_on='name_raw', right_on=name_col, how='inner')
    if len(merged) < 20: continue
    r_wobacon = merged[yr_col].corr(merged['actual_xwobacon'])
    print(f'\n  {yr} (n={len(merged)}):')
    print(f'    {yr} iSwing+ vs {yr} xwOBAcon   r={r_wobacon:+.4f}  R²={r_wobacon**2:.4f}')

print('\n' + '=' * 60)
print('PREDICTIVE (year N iSwing+ vs year N+1 xwOBAcon)')
print('=' * 60)
for yr in sorted(actual['year'].unique()):
    yr = int(yr)
    next_yr = yr + 1
    yr_col = str(yr)
    if yr_col not in lb.columns: continue
    next_actual = actual[actual['year'] == next_yr]
    if len(next_actual) == 0: continue
    merged = lb[lb[yr_col].notna()].merge(next_actual, left_on='name_raw', right_on=name_col, how='inner')
    if len(merged) < 20: continue
    r = merged[yr_col].corr(merged['actual_xwobacon'])
    print(f'\n  {yr} iSwing+ vs {next_yr} xwOBAcon (n={len(merged)}):')
    print(f'    r={r:+.4f}  R²={r**2:.4f}')

print('\n' + '=' * 60)
print('STABILITY (year N xwOBAcon vs year N+1 xwOBAcon)')
print('=' * 60)
for yr in sorted(actual['year'].unique()):
    yr = int(yr)
    next_yr = yr + 1
    a1 = actual[actual['year'] == yr][[name_col, 'actual_xwobacon']].rename(columns={'actual_xwobacon': 'now'})
    a2 = actual[actual['year'] == next_yr][[name_col, 'actual_xwobacon']].rename(columns={'actual_xwobacon': 'next'})
    if len(a2) == 0: continue
    m = a1.merge(a2, on=name_col, how='inner')
    if len(m) < 20: continue
    r = m['now'].corr(m['next'])
    print(f'  {yr} xwOBAcon vs {next_yr} xwOBAcon:  r={r:+.4f}  R²={r**2:.4f}  (n={len(m)})')